In [16]:
from SDP_interaction_inference.dataset import SparseDataset
import SDP_interaction_inference.utils
import anndata as ad
import numpy as np
import scipy
import tqdm

In [7]:
# load pcRNA
adata_pcRNA = ad.read_h5ad("../../TotalX/HEK293T/TotalX_HEK293T_pcRNA.h5ad")

# load miRNA
adata_miRNA = ad.read_h5ad("../../TotalX/HEK293T/TotalX_HEK293T_miRNA.h5ad")

# load capture
beta = np.loadtxt("../../TotalX/HEK293T/TotalX_HEK293T_capture.txt")

In [110]:
# setup dataset
dataset_SDP = SparseDataset()

# selection
miRNA_index = 5
G = 100

# construct dataset of miRNA paired with mRNA
dataset_SDP.construct_dataset(
    adata_miRNA[:, miRNA_index],
    adata_pcRNA[:, :G],
    beta
)

# Existing package

In [111]:
# settings
d = 3

# bootstrap
dataset_SDP.bootstrap(
    d=d,
    tqdm_disable=False
)

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [02:11<00:00,  1.31s/it]


# Deconstructed

In [11]:
def sparse_bootstrap(dataset, sample):
    '''
    Compute confidence intervals on the moments of a sample of count pairs, use
    resamples number of bootstrap resamples (default to sample size) and estimate
    moments up to order d.

    Args:
        dataset: stores settings such as
            d: maximum moment order to estimate
            confidence: confidence level of intervals
            resamples: integer number of bootstrap resamples to use
        sample: n (cells) x 2 sparse matrix of integer count pairs per cell

    Returns:
        (2 x Nd) numpy array of CI bounds on each Nd moment of order <= d
    '''

    # bootstrap to N x n x 2 array
    boot = dataset.rng.choice(sample.toarray(), size=(dataset.resamples, dataset.cells))

    # split into 2 N x n arrays 
    x1_boot = boot[:, :, 0]
    x2_boot = boot[:, :, 1]

    # estimate
    moment_bounds = np.zeros((2, dataset.Nd))
    for i, alpha in enumerate(dataset.powers):

        # raise boot to powers
        x1_boot_alpha = x1_boot**alpha[0]
        x2_boot_alpha = x2_boot**alpha[1]

        # multiply (N x n)
        boot_alpha = x1_boot_alpha * x2_boot_alpha

        # mean over sample axis (N x 1)
        moment_estimates = np.mean(boot_alpha, axis=1)

        # quantile over boot axis (2 x 1)
        alpha = 1 - dataset.confidence
        moment_interval = np.quantile(moment_estimates, [(alpha / 2), 1 - (alpha / 2)])

        # store
        moment_bounds[:, i] = moment_interval

    return moment_bounds

In [ ]:
def bootstrap(dataset, d, confidence=0.95, resamples=1000, tqdm_disable=True):
    '''
    For each sample in dataset compute bootstrap CI bounds on moments.

    Args:
        d: maximum moment order to estimate
    '''

    # store
    dataset.d = d
    dataset.confidence = confidence
    dataset.resamples = resamples

    # helpful objects
    dataset.rng = np.random.default_rng()
    dataset.powers = SDP_interaction_inference.utils.compute_powers(S=2, d=d)
    dataset.Nd = SDP_interaction_inference.utils.compute_Nd(S=2, d=d)

    # convert to sparse column array for faster slicing
    Xmi = dataset.miRNA_dataset.tocsc()
    Xpc = dataset.pcRNA_dataset.tocsc()

    # loop over pairds
    for i, pair in tqdm.tqdm(enumerate(dataset.pair_indices), disable=tqdm_disable, total=len(dataset.pair_indices)):

        # select samples
        miRNA_sample = Xmi[:, pair[0]]
        pcRNA_sample = Xpc[:, pair[1]]

        # combine
        sample = scipy.sparse.hstack([miRNA_sample, pcRNA_sample])

        # bootstrap moments
        moments = sparse_bootstrap(
            dataset,
            sample
        )

        # store moments
        dataset.moment_bounds[f'sample-{i}'] = moments

In [17]:
# bootstrap
bootstrap(
    dataset=dataset_SDP,
    d=d,
    tqdm_disable=False
)

100%|██████████| 100/100 [02:18<00:00,  1.38s/it]


# Pre-compute powers & sample indices

In [ ]:
def sparse_bootstrap_2(dataset, sample):
    '''
    Compute confidence intervals on the moments of a sample of count pairs, use
    resamples number of bootstrap resamples (default to sample size) and estimate
    moments up to order d.

    Args:
        dataset: stores settings such as
            d: maximum moment order to estimate
            confidence: confidence level of intervals
            resamples: integer number of bootstrap resamples to use
        sample: n (cells) x 2 sparse matrix of integer count pairs per cell

    Returns:
        (2 x Nd) numpy array of CI bounds on each Nd moment of order <= d
    '''

    # pre-compute powers
    x1_powers = [sample[:, 0]]
    x2_powers = [sample[:, 1]]
    for i in range(1, dataset.d + 1):
        x1_powers.append(sample[:, 0].power(i))
        x2_powers.append(sample[:, 1].power(i))

    # size
    N = dataset.cells
    B = dataset.resamples

    # sample bootstrap indices: B x N
    boot_indices = dataset.rng.integers(0, N, size=(B, N))

    # compute bootstrap weights
    boot_weights = np.zeros((B, N))
    for b in range(B):
        boot_weights[b, :] = np.bincount(boot_indices[b, :], minlength=N) / N

    # estimate
    moment_bounds = np.zeros((2, dataset.Nd))
    for i, alpha in enumerate(dataset.powers):

        # select powers
        x1 = x1_powers[alpha[0]]
        x2 = x2_powers[alpha[1]]

        # product
        x = x1.multiply(x2)

        # compute moments for each bootstrap resample
        moment_estimates = boot_weights @ x

        # quantile over boot axis (2 x 1)
        alpha = 1 - dataset.confidence
        moment_interval = np.quantile(moment_estimates, [(alpha / 2), 1 - (alpha / 2)])

        # store
        moment_bounds[:, i] = moment_interval

    return moment_bounds

In [39]:
def bootstrap_2(dataset, d, confidence=0.95, resamples=1000, tqdm_disable=True):
    '''
    For each sample in dataset compute bootstrap CI bounds on moments.

    Args:
        d: maximum moment order to estimate
    '''

    # store
    dataset.d = d
    dataset.confidence = confidence
    dataset.resamples = resamples

    # helpful objects
    dataset.rng = np.random.default_rng()
    dataset.powers = SDP_interaction_inference.utils.compute_powers(S=2, d=d)
    dataset.Nd = SDP_interaction_inference.utils.compute_Nd(S=2, d=d)

    # convert to sparse column array for faster slicing
    Xmi = dataset.miRNA_dataset.tocsc()
    Xpc = dataset.pcRNA_dataset.tocsc()

    # loop over pairds
    for i, pair in tqdm.tqdm(enumerate(dataset.pair_indices), disable=tqdm_disable, total=len(dataset.pair_indices)):

        # select samples
        miRNA_sample = Xmi[:, pair[0]]
        pcRNA_sample = Xpc[:, pair[1]]

        # combine
        sample = scipy.sparse.hstack([miRNA_sample, pcRNA_sample])

        # bootstrap moments
        moments = sparse_bootstrap_2(
            dataset,
            sample
        )

        # store moments
        dataset.moment_bounds[f'sample-{i}'] = moments

In [49]:
# bootstrap
bootstrap_2(
    dataset=dataset_SDP,
    d=d,
    tqdm_disable=False
)

100%|██████████| 100/100 [01:33<00:00,  1.07it/s]


# Pre-compute over all

In [46]:
def sparse_bootstrap_3(dataset, x1_powers, x2_powers):
    '''
    Compute confidence intervals on the moments of a sample of count pairs, use
    resamples number of bootstrap resamples (default to sample size) and estimate
    moments up to order d.

    Args:
        dataset: stores settings such as
            d: maximum moment order to estimate
            confidence: confidence level of intervals
            resamples: integer number of bootstrap resamples to use
        sample: n (cells) x 2 sparse matrix of integer count pairs per cell

    Returns:
        (2 x Nd) numpy array of CI bounds on each Nd moment of order <= d
    '''

    # size
    N = dataset.cells
    B = dataset.resamples

    # sample bootstrap indices: B x N
    boot_indices = dataset.rng.integers(0, N, size=(B, N))

    # compute bootstrap weights
    boot_weights = np.zeros((B, N))
    for b in range(B):
        boot_weights[b, :] = np.bincount(boot_indices[b, :], minlength=N) / N

    # estimate
    moment_bounds = np.zeros((2, dataset.Nd))
    for i, alpha in enumerate(dataset.powers):

        # select powers
        x1 = x1_powers[alpha[0]]
        x2 = x2_powers[alpha[1]]

        # product
        x = x1.multiply(x2)

        # compute moments for each bootstrap resample
        moment_estimates = boot_weights @ x

        # quantile over boot axis (2 x 1)
        alpha = 1 - dataset.confidence
        moment_interval = np.quantile(moment_estimates, [(alpha / 2), 1 - (alpha / 2)])

        # store
        moment_bounds[:, i] = moment_interval

    return moment_bounds

In [47]:
def bootstrap_3(dataset, d, confidence=0.95, resamples=1000, tqdm_disable=True):
    '''
    For each sample in dataset compute bootstrap CI bounds on moments.

    Args:
        d: maximum moment order to estimate
    '''

    # store
    dataset.d = d
    dataset.confidence = confidence
    dataset.resamples = resamples

    # helpful objects
    dataset.rng = np.random.default_rng()
    dataset.powers = SDP_interaction_inference.utils.compute_powers(S=2, d=d)
    dataset.Nd = SDP_interaction_inference.utils.compute_Nd(S=2, d=d)

    # convert to sparse column array for faster slicing
    Xmi = dataset.miRNA_dataset.tocsc()
    Xpc = dataset.pcRNA_dataset.tocsc()

    # compute powers
    Xmi_powers = [Xmi]
    Xpc_powers = [Xpc]
    for i in range(1, dataset.d + 1):
        Xmi_powers.append(Xmi.power(i))
        Xpc_powers.append(Xpc.power(i))

    # store
    dataset.Xmi_powers = Xmi_powers
    dataset.Xpc_powers = Xpc_powers

    # loop over pairs
    for i, pair in tqdm.tqdm(enumerate(dataset.pair_indices), disable=tqdm_disable, total=len(dataset.pair_indices)):

        # select samples
        x1_powers = [x1[:, pair[0]] for x1 in Xmi_powers]
        x2_powers = [x2[:, pair[1]] for x2 in Xpc_powers]

        # bootstrap moments
        moments = sparse_bootstrap_3(
            dataset,
            x1_powers,
            x2_powers
        )

        # store moments
        dataset.moment_bounds[f'sample-{i}'] = moments

In [48]:
# bootstrap
bootstrap_3(
    dataset=dataset_SDP,
    d=d,
    tqdm_disable=False
)

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [01:27<00:00,  1.14it/s]


# Full vectorization

In [72]:
import SDP_interaction_inference.utils

In [114]:
def compute_all_cis(query_moments, alpha=0.05):
    """
    Computes lower and upper percentile confidence intervals 
    for all 10,000 queries simultaneously.
    """
    low_p = (alpha / 2) * 100
    high_p = (1 - alpha / 2) * 100
    
    # Vectorized percentile call across axis=2 (the bootstrap replicates)
    ci_lower, ci_upper = np.percentile(query_moments, [low_p, high_p], axis=2)
    
    return ci_lower, ci_upper

## miRNA assumed constant

In [86]:
def compute_all_multivariate_query_moments(sparse_A, sparse_B, shared_A_species, queries_B_species, max_order=3, num_bootstraps=1000, chunk_size=100):
    """
    Computes ALL multivariate moments up to order d for each query.
    Optimized for constant A selection and variable B selection.
    """
    N, _ = sparse_A.shape
    
    # already csr
    A_csr = scipy.sparse.csr_matrix(sparse_A)
    B_csr = scipy.sparse.csr_matrix(sparse_B)
    
    # 1. Pre-extract the constant A species columns
    # Shape: (N, len(shared_A_species))
    A_base = np.hstack([A_csr[:, sp].toarray() for sp in shared_A_species])
    num_A = len(shared_A_species)
    
    # 2. Pre-generate the power combinations matrix
    # Assumes each query has the same structure: len(shared_A_species) + len(b_list)
    # If len(b_list) varies slightly, generate for the maximum length and pad with 0s.
    sample_B_len = len(queries_B_species[0])
    total_vars = num_A + sample_B_len
    
    # Exponent grid shape: (num_moments, total_vars)
    exponent_grid = np.array(SDP_interaction_inference.utils.compute_powers(total_vars, max_order), dtype=np.int32) # generate_valid_powers(total_vars, max_order)
    num_moments_per_query = exponent_grid.shape[0]
    
    # 3. Pre-allocate the complete master Q_matrix
    # Shape: (N, total_queries * num_moments_per_query)
    total_queries = len(queries_B_species)
    Q_matrix = np.zeros((N, total_queries * num_moments_per_query), dtype=np.float64)
    
    # 4. Construct Q_matrix via vectorized power broadcasting
    for q_idx, b_list in enumerate(queries_B_species):
        # Extract B species for this query: shape (N, num_B)
        B_base = np.hstack([B_csr[:, sp].toarray() for sp in b_list])
        
        # Combine base variables: shape (N, total_vars)
        combined_base = np.hstack([A_base, B_base])
        
        # Vectorized expansion: Broadcast base variables over the exponent grid
        # base[:, np.newaxis, :] -> (N, 1, total_vars)
        # exponents[np.newaxis, :, :] -> (1, num_moments, total_vars)
        # Resulting power shape: (N, num_moments, total_vars)
        powers_tensor = np.power(combined_base[:, np.newaxis, :], exponent_grid[np.newaxis, :, :])
        
        # Product along the species axis (axis=2) to get final moments for this query
        # Shape: (N, num_moments)
        query_all_moments = np.prod(powers_tensor, axis=2)
        
        # Map straight into our slice of the master Q_matrix
        start_col = q_idx * num_moments_per_query
        end_col = start_col + num_moments_per_query
        Q_matrix[:, start_col:end_col] = query_all_moments

    # 5. Execute Bootstrap Resampling via standard high-speed BLAS dot product
    # Output array shape: (total_queries * num_moments_per_query, num_bootstraps)
    final_moments_flat = np.zeros((total_queries * num_moments_per_query, num_bootstraps), dtype=np.float64)
    boot_indices = np.random.randint(0, N, size=(N, num_bootstraps))
    
    for start_idx in range(0, num_bootstraps, chunk_size):
        end_idx = min(start_idx + chunk_size, num_bootstraps)
        current_chunk_size = end_idx - start_idx
        
        chunk_inds = boot_indices[:, start_idx:end_idx]
        
        weights = np.zeros((N, current_chunk_size), dtype=np.float64)
        for col in range(current_chunk_size):
            weights[:, col] = np.bincount(chunk_inds[:, col], minlength=N) / N
            
        final_moments_flat[:, start_idx:end_idx] = np.dot(Q_matrix.T, weights)
        
    # 6. Reshape into an easily queryable multi-dimensional tensor
    # Shape: (total_queries, num_moments_per_query, num_bootstraps)
    final_moments = final_moments_flat.reshape(total_queries, num_moments_per_query, num_bootstraps)

    return final_moments, exponent_grid

In [104]:
def bootstrap_4(dataset, d, confidence=0.95, resamples=1000, tqdm_disable=True):
    '''
    For each sample in dataset compute bootstrap CI bounds on moments.

    Args:
        d: maximum moment order to estimate
    '''

    sparse_A = dataset.miRNA_dataset
    sparse_B = dataset.pcRNA_dataset
    shared_A_species = [0]
    queries_B_species = [[pair[1]] for pair in dataset.pair_indices]

    final_moments, exponent_grid = compute_all_multivariate_query_moments(
        sparse_A,
        sparse_B,
        shared_A_species,
        queries_B_species,
        max_order=3,
        num_bootstraps=1000,
        chunk_size=100
    )

    lb, ub = compute_all_cis(final_moments)

    return lb, ub

In [105]:
# bootstrap
moment_lb, moment_ub = bootstrap_4(
    dataset=dataset_SDP,
    d=d,
    tqdm_disable=False
)

In [ ]:
moment_lb[0, :]

array([  1.        ,   2.81362913,   0.98064881,  15.47203014,
         3.96282545,   2.54345961, 116.83415027,  26.43273545,
        11.9261825 ,   9.26047509])

In [113]:
moment_ub[0, :]

array([  1.        ,   2.91295103,   1.02436584,  16.61226245,
         4.27673713,   2.79415655, 132.31933864,  30.11314776,
        14.19867099,  11.81391377])

In [112]:
dataset_SDP.moment_bounds['sample-0']

array([[  1.        ,   2.81413772,   0.97839891,  15.46323946,
          3.95007123,   2.53963367, 116.7072588 ,  26.39341908,
         11.97426729,   9.26850989],
       [  1.        ,   2.91294477,   1.02478655,  16.56275234,
          4.26371487,   2.79930515, 131.40741386,  29.98644967,
         14.10999753,  11.81061325]])

## miRNA can vary

In [120]:
def compute_unshared_query_moments(sparse_A, sparse_B, queries_A_list, queries_B_list, max_order=3, num_bootstraps=1000, chunk_size=100):
    """
    Optimized for queries where BOTH A and B species selections change per query,
    but the number of species per query remains constant.
    
    Parameters:
    - sparse_A, sparse_B: Scipy sparse matrices (N x Species)
    - queries_A_list: List of lists containing A species indices per query, e.g., [,]
    - queries_B_list: List of lists containing B species indices per query, e.g., [,]
    """
    N, _ = sparse_A.shape
    total_queries = len(queries_A_list)
    
    num_A = len(queries_A_list[0])
    num_B = len(queries_B_list[0])
    total_vars = num_A + num_B
    
    # 1. Pre-generate exponent grid
    exponent_grid = generate_valid_powers(total_vars, max_order)
    num_moments = exponent_grid.shape[0]
    
    # 2. MASSIVE OPTIMIZATION: Flatten queries to extract all data in ONE step
    # This completely bypasses the Scipy sparse slicing loop bottleneck.
    flat_A_indices = np.array(queries_A_list).flatten()
    flat_B_indices = np.array(queries_B_list).flatten()
    
    A_csr = scipy.sparse.csr_matrix(sparse_A)
    B_csr = scipy.sparse.csr_matrix(sparse_B)
    
    # Extract everything to dense arrays in single operations
    # Shapes: (N, total_queries * num_A) and (N, total_queries * num_B)
    all_A_data = A_csr[:, flat_A_indices].toarray()
    all_B_data = B_csr[:, flat_B_indices].toarray()
    
    # Reshape into 3D arrays: (N, total_queries, num_features)
    all_A_data = all_A_data.reshape(N, total_queries, num_A)
    all_B_data = all_B_data.reshape(N, total_queries, num_B)
    
    # Combine them along the feature axis -> Shape: (N, total_queries, total_vars)
    combined_base = np.concatenate([all_A_data, all_B_data], axis=2)
    
    # 3. Pre-allocate the master Q_matrix
    Q_matrix = np.zeros((N, total_queries * num_moments), dtype=np.float64)
    exponents_broadcasted = exponent_grid[np.newaxis, np.newaxis, :, :] # Shape: (1, 1, num_moments, total_vars)
    
    # 4. Construct Q_matrix via vectorized power broadcasting
    # We process in chunks of queries to avoid creating a massive 4D tensor that breaks RAM
    query_chunk_size = 500  # Adjust based on available system RAM
    for q_start in range(0, total_queries, query_chunk_size):
        q_end = min(q_start + query_chunk_size, total_queries)
        curr_q_len = q_end - q_start
        
        # Slice out a chunk of queries: (N, curr_q_len, total_vars)
        base_chunk = combined_base[:, q_start:q_end, :]
        
        # Broadcast over exponents array:
        # (N, curr_q_len, 1, total_vars) ** (1, 1, num_moments, total_vars)
        # Tensor Shape: (N, curr_q_len, num_moments, total_vars)
        powers_tensor = np.power(base_chunk[:, :, np.newaxis, :], exponents_broadcasted)
        
        # Product along the species/variable axis (axis=3) -> Shape: (N, curr_q_len, num_moments)
        query_all_moments = np.prod(powers_tensor, axis=3)
        
        # Flatten the query and moment dimensions to fit into our 2D Q_matrix
        # Reshapes (N, curr_q_len, num_moments) -> (N, curr_q_len * num_moments)
        flattened_moments = query_all_moments.reshape(N, curr_q_len * num_moments)
        
        # Map straight into the pre-allocated master matrix
        col_start = q_start * num_moments
        col_end = q_end * num_moments
        Q_matrix[:, col_start:col_end] = flattened_moments

    # 5. Core Bootstrap Pipeline (BLAS Dot Product)
    final_moments_flat = np.zeros((total_queries * num_moments, num_bootstraps), dtype=np.float64)
    boot_indices = np.random.randint(0, N, size=(N, num_bootstraps))
    
    for start_idx in range(0, num_bootstraps, chunk_size):
        end_idx = min(start_idx + chunk_size, num_bootstraps)
        current_chunk_size = end_idx - start_idx
        
        chunk_inds = boot_indices[:, start_idx:end_idx]
        
        weights = np.zeros((N, current_chunk_size), dtype=np.float64)
        for col in range(current_chunk_size):
            weights[:, col] = np.bincount(chunk_inds[:, col], minlength=N) / N
            
        final_moments_flat[:, start_idx:end_idx] = np.dot(Q_matrix.T, weights)
        
    # Reshape back to cleanly structured final dimensions
    # Shape: (total_queries, num_moments, num_bootstraps)
    final_moments = final_moments_flat.reshape(total_queries, num_moments, num_bootstraps)

    return final_moments, exponent_grid

In [118]:
def bootstrap_5(dataset, d, confidence=0.95, resamples=1000, tqdm_disable=True):
    '''
    For each sample in dataset compute bootstrap CI bounds on moments.

    Args:
        d: maximum moment order to estimate
    '''

    sparse_A = dataset.miRNA_dataset
    sparse_B = dataset.pcRNA_dataset
    queries_A_list = [[pair[0]] for pair in dataset.pair_indices]
    queries_B_list = [[pair[1]] for pair in dataset.pair_indices]
    
    final_moments, exponent_grid = compute_unshared_query_moments(
        sparse_A,
        sparse_B,
        queries_A_list,
        queries_B_list,
        max_order=3,
        num_bootstraps=1000,
        chunk_size=100
    )

    lb, ub = compute_all_cis(final_moments)

    return lb, ub

In [121]:
# bootstrap
moment_lb, moment_ub = bootstrap_5(
    dataset=dataset_SDP,
    d=d,
    tqdm_disable=False
)

In [124]:
moment_lb[0, :]

array([  1.        ,   2.81289033,   0.97948723,  15.48821683,
         3.94916283,   2.53819799, 117.11905609,  26.39588112,
        11.90001465,   9.27351821])

In [125]:
moment_ub[0, :]

array([  1.        ,   2.91203014,   1.02377982,  16.55889284,
         4.28407493,   2.79844077, 131.35636668,  30.07524278,
        14.15648807,  11.73593554])

In [126]:
dataset_SDP.moment_bounds['sample-0']

array([[  1.        ,   2.81413772,   0.97839891,  15.46323946,
          3.95007123,   2.53963367, 116.7072588 ,  26.39341908,
         11.97426729,   9.26850989],
       [  1.        ,   2.91294477,   1.02478655,  16.56275234,
          4.26371487,   2.79930515, 131.40741386,  29.98644967,
         14.10999753,  11.81061325]])

## Tidy

In [ ]:
def bootstrap_6(dataset, d, confidence=0.95, resamples=1000, chunk_size=100):
    """
    Optimized for queries where BOTH A and B species selections change per query,
    but the number of species per query remains constant.
    
    Parameters:
    - sparse_A, sparse_B: Scipy sparse matrices (N x Species)
    - queries_A_list: List of lists containing A species indices per query, e.g., [,]
    - queries_B_list: List of lists containing B species indices per query, e.g., [,]
    """

    # store
    dataset.d = d
    dataset.confidence = confidence
    dataset.resamples = resamples

    # collect data
    sparse_A = dataset.miRNA_dataset
    sparse_B = dataset.pcRNA_dataset
    queries_A_list = [[pair[0]] for pair in dataset.pair_indices]
    queries_B_list = [[pair[1]] for pair in dataset.pair_indices]
    max_order = dataset.d
    num_bootstraps = dataset.resamples

    # size
    N, _ = sparse_A.shape
    total_queries = len(queries_A_list)
    
    num_A = len(queries_A_list[0])
    num_B = len(queries_B_list[0])
    total_vars = num_A + num_B
    
    # 1. Pre-generate exponent grid
    exponent_grid = np.array(SDP_interaction_inference.utils.compute_powers(total_vars, max_order), dtype=np.int32)
    num_moments = exponent_grid.shape[0]
    
    # 2. MASSIVE OPTIMIZATION: Flatten queries to extract all data in ONE step
    # This completely bypasses the Scipy sparse slicing loop bottleneck.
    flat_A_indices = np.array(queries_A_list).flatten()
    flat_B_indices = np.array(queries_B_list).flatten()
    
    A_csr = scipy.sparse.csr_matrix(sparse_A)
    B_csr = scipy.sparse.csr_matrix(sparse_B)
    
    # Extract everything to dense arrays in single operations
    # Shapes: (N, total_queries * num_A) and (N, total_queries * num_B)
    all_A_data = A_csr[:, flat_A_indices].toarray()
    all_B_data = B_csr[:, flat_B_indices].toarray()
    
    # Reshape into 3D arrays: (N, total_queries, num_features)
    all_A_data = all_A_data.reshape(N, total_queries, num_A)
    all_B_data = all_B_data.reshape(N, total_queries, num_B)
    
    # Combine them along the feature axis -> Shape: (N, total_queries, total_vars)
    combined_base = np.concatenate([all_A_data, all_B_data], axis=2)
    
    # 3. Pre-allocate the master Q_matrix
    Q_matrix = np.zeros((N, total_queries * num_moments), dtype=np.float64)
    exponents_broadcasted = exponent_grid[np.newaxis, np.newaxis, :, :] # Shape: (1, 1, num_moments, total_vars)
    
    # 4. Construct Q_matrix via vectorized power broadcasting
    # We process in chunks of queries to avoid creating a massive 4D tensor that breaks RAM
    query_chunk_size = 500  # Adjust based on available system RAM
    for q_start in range(0, total_queries, query_chunk_size):
        q_end = min(q_start + query_chunk_size, total_queries)
        curr_q_len = q_end - q_start
        
        # Slice out a chunk of queries: (N, curr_q_len, total_vars)
        base_chunk = combined_base[:, q_start:q_end, :]
        
        # Broadcast over exponents array:
        # (N, curr_q_len, 1, total_vars) ** (1, 1, num_moments, total_vars)
        # Tensor Shape: (N, curr_q_len, num_moments, total_vars)
        powers_tensor = np.power(base_chunk[:, :, np.newaxis, :], exponents_broadcasted)
        
        # Product along the species/variable axis (axis=3) -> Shape: (N, curr_q_len, num_moments)
        query_all_moments = np.prod(powers_tensor, axis=3)
        
        # Flatten the query and moment dimensions to fit into our 2D Q_matrix
        # Reshapes (N, curr_q_len, num_moments) -> (N, curr_q_len * num_moments)
        flattened_moments = query_all_moments.reshape(N, curr_q_len * num_moments)
        
        # Map straight into the pre-allocated master matrix
        col_start = q_start * num_moments
        col_end = q_end * num_moments
        Q_matrix[:, col_start:col_end] = flattened_moments

    # 5. Core Bootstrap Pipeline (BLAS Dot Product)
    final_moments_flat = np.zeros((total_queries * num_moments, num_bootstraps), dtype=np.float64)
    boot_indices = np.random.randint(0, N, size=(N, num_bootstraps))
    
    for start_idx in range(0, num_bootstraps, chunk_size):
        end_idx = min(start_idx + chunk_size, num_bootstraps)
        current_chunk_size = end_idx - start_idx
        
        chunk_inds = boot_indices[:, start_idx:end_idx]
        
        weights = np.zeros((N, current_chunk_size), dtype=np.float64)
        for col in range(current_chunk_size):
            weights[:, col] = np.bincount(chunk_inds[:, col], minlength=N) / N
            
        final_moments_flat[:, start_idx:end_idx] = np.dot(Q_matrix.T, weights)
        
    # Reshape back to cleanly structured final dimensions
    # Shape: (total_queries, num_moments, num_bootstraps)
    final_moments = final_moments_flat.reshape(total_queries, num_moments, num_bootstraps)

    # compute percentile confidence intervals over bootstrap axis
    alpha = 1 - dataset.confidence
    moment_interval = np.quantile(final_moments, [(alpha / 2), 1 - (alpha / 2)], axis=2)

    return moment_interval

In [ ]:
# bootstrap
moment_interval = bootstrap_6(
    dataset=dataset_SDP,
    d=d
)

In [150]:
g = 1
np.vstack([moment_interval[0, g, :], moment_interval[1, g, :]])

array([[  1.        ,   2.81531185,   1.58744035,  15.48252198,
          6.76939096,   5.8761846 , 117.2021013 ,  45.61994768,
         30.01832357,  30.20635622],
       [  1.        ,   2.91379657,   1.65166806,  16.56490791,
          7.24294684,   6.34434073, 131.54878192,  50.85198619,
         33.56184177,  34.38320427]])

In [151]:
dataset_SDP.moment_bounds[f'sample-{g}']

array([[  1.        ,   2.81280038,   1.58660322,  15.44134188,
          6.7593073 ,   5.88841971, 116.35406857,  45.63511438,
         30.07524762,  30.3775804 ],
       [  1.        ,   2.91429886,   1.65493308,  16.59069567,
          7.25027635,   6.36050439, 131.99151421,  50.89183455,
         33.62545395,  34.42864447]])

In [152]:
# setup dataset
dataset_SDP = SparseDataset()

# selection
miRNA_index = 5
G = 11000

# construct dataset of miRNA paired with mRNA
dataset_SDP.construct_dataset(
    adata_miRNA[:, miRNA_index],
    adata_pcRNA[:, :G],
    beta
)

In [ ]:
# settings
d = 3

# bootstrap
moment_interval = bootstrap_6(
    dataset=dataset_SDP,
    d=d
)

MemoryError: Unable to allocate 9.17 GiB for an array with shape (11945, 103070) and data type float64

# Memory: slice gene selections rathers than doing all at once

In [ ]:
def bootstrap_7(dataset, d, resamples=1000, confidence=0.95, query_chunk_size=100, bootstrap_chunk_size=100):
    """
    Computes percentile confidence intervals directly within localized chunks.
    Peak memory footprint stays ultra-low because the bootstrap matrix is discarded instantly.
    
    Returns:
    - ci_lower: 2D array of shape (Total Queries, Num Moments)
    - ci_upper: 2D array of shape (Total Queries, Num Moments)
    - exponent_grid: Mapping array showing which power combination each moment index represents
    """

    # store
    dataset.d = d
    dataset.confidence = confidence
    dataset.resamples = resamples

    # collect data
    sparse_A = dataset.miRNA_dataset
    sparse_B = dataset.pcRNA_dataset
    queries_A_list = [[pair[0]] for pair in dataset.pair_indices]
    queries_B_list = [[pair[1]] for pair in dataset.pair_indices]
    max_order = dataset.d
    num_bootstraps = dataset.resamples

    # size
    N, _ = sparse_A.shape
    total_queries = len(queries_A_list)
    
    num_A = len(queries_A_list[0])
    num_B = len(queries_B_list[0])
    total_vars = num_A + num_B
    
    # 1. Generate exponent grid (constant across all queries)
    exponent_grid = np.array(SDP_interaction_inference.utils.compute_powers(total_vars, max_order), dtype=np.int32)
    num_moments = exponent_grid.shape[0]
    
    # alpha
    alpha = 1 - dataset.confidence
    
    # 2. Convert to CSR format once
    A_csr = scipy.sparse.csr_matrix(sparse_A)
    B_csr = scipy.sparse.csr_matrix(sparse_B)
    
    # 3. Generate unified bootstrap indices up front
    boot_indices = np.random.randint(0, N, size=(N, num_bootstraps))
    
    # 4. Pre-allocate ONLY the final target tables (Tiny! Shape: 10000 x 35)
    ci_lower = np.zeros((total_queries, num_moments), dtype=np.float64)
    ci_upper = np.zeros((total_queries, num_moments), dtype=np.float64)
    
    exponents_broadcasted = exponent_grid[np.newaxis, np.newaxis, :, :] # (1, 1, num_moments, total_vars)
    
    # 5. Global Loop: Process blocks of queries independently
    for q_start in range(0, total_queries, query_chunk_size):
        print(f"Queries {round(100 * q_start / total_queries, 2)}%")
        q_end = min(q_start + query_chunk_size, total_queries)
        curr_q_len = q_end - q_start
        
        chunk_queries_A = queries_A_list[q_start:q_end]
        chunk_queries_B = queries_B_list[q_start:q_end]
        
        flat_A_indices = np.array(chunk_queries_A).flatten()
        flat_B_indices = np.array(chunk_queries_B).flatten()
        
        # Batch extract and reshape data for this localized block
        all_A_data = A_csr[:, flat_A_indices].toarray().reshape(N, curr_q_len, num_A)
        all_B_data = B_csr[:, flat_B_indices].toarray().reshape(N, curr_q_len, num_B)
        combined_base = np.concatenate([all_A_data, all_B_data], axis=2)
        
        # Create small, transient Q_matrix just for this query block
        local_Q_matrix = np.zeros((N, curr_q_len * num_moments), dtype=np.float64)
        
        # Vectorised power calculation
        powers_tensor = np.power(combined_base[:, :, np.newaxis, :], exponents_broadcasted)
        query_all_moments = np.prod(powers_tensor, axis=3)
        local_Q_matrix[:, :] = query_all_moments.reshape(N, curr_q_len * num_moments)
        
        # 6. Embedded Bootstrap Loop for this local query block
        local_moments_flat = np.zeros((curr_q_len * num_moments, num_bootstraps), dtype=np.float64)
        
        for b_start in range(0, num_bootstraps, bootstrap_chunk_size):
            b_end = min(b_start + bootstrap_chunk_size, num_bootstraps)
            curr_b_len = b_end - b_start
            
            chunk_inds = boot_indices[:, b_start:b_end]
            
            # Generate local bootstrap weights
            weights = np.zeros((N, curr_b_len), dtype=np.float64)
            for col in range(curr_b_len):
                weights[:, col] = np.bincount(chunk_inds[:, col], minlength=N) / N
                
            local_moments_flat[:, b_start:b_end] = np.dot(local_Q_matrix.T, weights)
            
        # Reshape local bootstrap matrix to (curr_q_len, num_moments, num_bootstraps)
        local_moments_3d = local_moments_flat.reshape(curr_q_len, num_moments, num_bootstraps)
        
        # 7. IMMEDIATE QUANTILE PROCESSING (The Memory Saver)
        # Calculates percentiles over the replicates axis (axis=2) instantly.
        # This completely drops the 3rd dimension from memory right after calculating.
        # compute percentile confidence intervals over bootstrap axis
        chunk_low, chunk_high = np.quantile(local_moments_3d, [(alpha / 2), 1 - (alpha / 2)], axis=2)
        
        # Store directly into the final lightweight master tables
        ci_lower[q_start:q_end, :] = chunk_low
        ci_upper[q_start:q_end, :] = chunk_high
        
    return ci_lower, ci_upper

In [177]:
# setup dataset
dataset_SDP = SparseDataset()

# selection
miRNA_index = 5
G = 100

# construct dataset of miRNA paired with mRNA
dataset_SDP.construct_dataset(
    adata_miRNA[:, miRNA_index],
    adata_pcRNA[:, :G],
    beta
)

In [178]:
# settings
d = 3

# bootstrap
moment_lb, moment_ub = bootstrap_7(
    dataset=dataset_SDP,
    d=d
)

Queries 0.0%


In [179]:
# setup dataset
dataset_SDP = SparseDataset()

# selection
miRNA_index = 5
G = 11000

# construct dataset of miRNA paired with mRNA
dataset_SDP.construct_dataset(
    adata_miRNA[:, miRNA_index],
    adata_pcRNA[:, :G],
    beta
)

In [180]:
# settings
d = 3

# bootstrap
moment_lb, moment_ub = bootstrap_7(
    dataset=dataset_SDP,
    d=d
)

Queries 0.0%
Queries 9.7%
Queries 19.4%
Queries 29.11%
Queries 38.81%
Queries 48.51%
Queries 58.21%
Queries 67.92%
Queries 77.62%
Queries 87.32%
Queries 97.02%


In [184]:
# settings
d = 3

# bootstrap
moment_lb, moment_ub = bootstrap_7(
    dataset=dataset_SDP,
    d=d,
    query_chunk_size=100
)

Queries 0.0%
Queries 0.97%
Queries 1.94%
Queries 2.91%
Queries 3.88%
Queries 4.85%
Queries 5.82%
Queries 6.79%
Queries 7.76%
Queries 8.73%
Queries 9.7%
Queries 10.67%
Queries 11.64%
Queries 12.61%
Queries 13.58%
Queries 14.55%
Queries 15.52%
Queries 16.49%
Queries 17.46%
Queries 18.43%
Queries 19.4%
Queries 20.37%
Queries 21.34%
Queries 22.31%
Queries 23.29%
Queries 24.26%
Queries 25.23%
Queries 26.2%
Queries 27.17%
Queries 28.14%
Queries 29.11%
Queries 30.08%
Queries 31.05%
Queries 32.02%
Queries 32.99%
Queries 33.96%
Queries 34.93%
Queries 35.9%
Queries 36.87%
Queries 37.84%
Queries 38.81%
Queries 39.78%
Queries 40.75%
Queries 41.72%
Queries 42.69%
Queries 43.66%
Queries 44.63%
Queries 45.6%
Queries 46.57%
Queries 47.54%
Queries 48.51%
Queries 49.48%
Queries 50.45%
Queries 51.42%
Queries 52.39%
Queries 53.36%
Queries 54.33%
Queries 55.3%
Queries 56.27%
Queries 57.24%
Queries 58.21%
Queries 59.18%
Queries 60.15%
Queries 61.12%
Queries 62.09%
Queries 63.06%
Queries 64.03%
Queries 65.0%

In [187]:
g = 1
np.vstack([moment_lb[g, :], moment_ub[g, :]])

array([[  1.        ,   2.81405609,   1.58467769,  15.45424236,
          6.75826496,   5.86998116, 116.90474466,  45.53729803,
         29.8969527 ,  30.13373587],
       [  1.        ,   2.91327752,   1.65332775,  16.59028882,
          7.2529594 ,   6.35471746, 132.01435119,  50.84197781,
         33.53482419,  34.37419422]])

# Summary

- pre-computing sample powers outside of bootstrap and using index / weight based bootstrap resampling (rather than actual resampling of values) gives ~2 x speedup
- AI fancy vectorized code gives ~60 x speedup
    - fine to use full generality A, B species choices, useful for mRNA - mRNA analysis